# Getting Started with Alpaca Paper Trading
### SIGecom Algo-Trading Project, Fall 2026

This notebook is your first hands-on look at **Alpaca**, the brokerage API we use for the project. Work through it on your own. By the end you will have:

1. Set up your own **Alpaca paper trading account** (fake money, real market data).
2. Generated API keys and stored them safely.
3. Connected to Alpaca from Python and checked your account.
4. Pulled historical and live market data.
5. Placed, checked, and cancelled orders, and managed positions.

**No trading or finance experience needed.** Most of the code is already written for you. Your job is to run it, read the comments, and make small changes.

> **Paper trading only.** Everything here uses simulated money. You don't need to deposit anything, and you should **not** enter bank details or fund a live account for this project.

---
## Part 1: What is Alpaca?

[Alpaca](https://alpaca.markets) is a commission-free brokerage built for programmatic trading. Instead of clicking buttons in an app, you send orders through a **REST API**. Alpaca gives every account two environments:

| | **Paper** | **Live** |
|---|---|---|
| Money | Simulated (starts at $100,000 by default) | Real |
| Market data | Real | Real |
| Order fills | Simulated against real quotes | Real exchange fills |
| API base URL | `https://paper-api.alpaca.markets` | `https://api.alpaca.markets` |
| API keys | Paper keys | Live keys (different!) |

Paper and live keys are **separate**. Paper keys can't touch real money, so they're the only keys you need for this project.

---
## Part 2: Create your paper trading account

Alpaca updates its dashboard from time to time, so button labels may be slightly different from what's written here. The steps stay the same.

### Step 1: Sign up
1. Go to **https://app.alpaca.markets/signup**.
2. Sign up with your email and a strong password. (Your school email is fine.)
3. Open the verification email from Alpaca and click the link to confirm your address.

A paper-only account needs nothing more than a verified email. **You do not need to finish the identity verification / KYC steps** (SSN, address, employment, and so on). Those are only for opening a *live* account. If the dashboard asks you to "complete your application," you can skip it.

### Step 2: Switch to your paper account
1. Log in at **https://app.alpaca.markets**.
2. Find the **account switcher** in the top-left corner of the dashboard. It shows either your live account or a paper account (paper account numbers start with **`PA`**).
3. Select your **Paper Trading** account. The dashboard should now show a balance of about **$100,000**.

> Always check that you're on the **paper** account before generating keys. Everything below assumes you are.

### Step 3: Generate API keys
1. On the paper account's home page, find the **API Keys** panel (usually on the right side).
2. Click **Generate New Keys** (or **Regenerate** if keys already exist).
3. You'll see two values:
   - **API Key ID**, for example `PKXXXXXXXXXXXXXXXXXX` (paper keys usually start with `PK`)
   - **Secret Key**, a longer random string
4. **Copy both right away.** Alpaca shows the Secret Key **only once**. If you lose it, regenerate the keys (which invalidates the old pair).

### Step 4 (optional): Reset or resize your paper account
From the paper account menu you can **reset** the account, which wipes positions and history and lets you choose a new starting balance. This helps if you want to start over clean or test with a realistic amount (say $10,000 instead of $100,000).

> Paper accounts follow the **Pattern Day Trader (PDT)** rule just like real ones: if equity drops below **$25,000**, you're limited to 3 day trades in any rolling 5 business days. Keep this in mind if you shrink your balance.

---
## Part 3: Keep your keys safe

API keys are passwords. Anyone who has them can trade in your account. Even though this is paper money, practice good habits now:

- **Never** paste keys directly into a notebook cell or `.py` file.
- **Never** commit keys to Git. Bots scrape GitHub for leaked keys within minutes.
- Store them in a **`.env`** file that Git ignores.

### Create a `.env` file
In the **same folder as this notebook**, create a file named exactly `.env` (no filename before the dot) with these contents, replacing the placeholders with your keys:

```
ALPACA_API_KEY=PKXXXXXXXXXXXXXXXXXX
ALPACA_SECRET_KEY=your_secret_key_here
```

No quotes, and no spaces around the `=`. This repo's `.gitignore` already ignores `.env`, so it won't be committed. You can copy `.env.example` to `.env` as a starting point.

> **Using Google Colab?** Skip this step. The notebook will ask you to paste your keys when it needs them.

---
## Part 4: Install the Python packages

We use **[`alpaca-py`](https://alpaca.markets/sdks/python/)**, Alpaca's official Python SDK.

> ⚠️ You may see older tutorials that use `alpaca-trade-api` or `import alpaca_trade_api`. That library is **deprecated**. Use `alpaca-py` (imported as `alpaca`).

Run the cell below once. If you use a virtual environment or conda env, make sure Jupyter is running inside it.

In [ ]:
# Install the libraries this notebook uses. You only need to run this once.
#   alpaca-py      -> talks to Alpaca
#   python-dotenv  -> reads your API keys from the .env file
#   pandas         -> works with tables of data
#   matplotlib     -> draws charts
%pip install --quiet alpaca-py python-dotenv pandas matplotlib

---
## Part 5: Load your keys and connect

### How to read the code in this notebook
- Lines starting with `#` are **comments**. Python ignores them; they're there to explain what's going on. Read them!
- Run a cell with **`Shift+Enter`**. Run cells **in order**, because later cells use things created by earlier ones.
- If you see a `NameError`, you probably skipped a cell above.

The cell below reads your keys from `.env`. If it can't find them, it asks you to type them in instead (they won't be saved anywhere).

In [ ]:
import os                       # lets Python read settings from your computer
from getpass import getpass     # asks you to type something without showing it on screen
from dotenv import load_dotenv  # reads the .env file

# Read your .env file, if you have one
load_dotenv()

# Look for each key in .env. If it isn't there, ask you to paste it in.
API_KEY = os.getenv("ALPACA_API_KEY") or getpass("Alpaca API Key ID: ")
SECRET_KEY = os.getenv("ALPACA_SECRET_KEY") or getpass("Alpaca Secret Key: ")

# Stop with a helpful message if either key is missing
assert API_KEY and SECRET_KEY, "Missing Alpaca keys. See Part 3."

# Only print the first 4 characters. Never print your full keys!
print("Keys loaded. API key starts with:", API_KEY[:4] + "...")

`alpaca-py` splits its features across several **clients** (objects that talk to Alpaca for you). We use two:

- **`trading_client`**: your account, orders, positions, and the market clock
- **`data_client`**: prices (bars, quotes)

`paper=True` makes sure every request goes to your **paper** account. **Always include it** in this project.

In [ ]:
from alpaca.trading.client import TradingClient
from alpaca.data.historical import StockHistoricalDataClient

# Create the two clients using your keys
trading_client = TradingClient(API_KEY, SECRET_KEY, paper=True)  # paper=True means fake money
data_client = StockHistoricalDataClient(API_KEY, SECRET_KEY)

### Check your account
If this cell runs without errors, your keys work. If you get a `401 Unauthorized` / `forbidden` error, the usual causes are:
- live keys instead of paper keys,
- a typo or extra whitespace in `.env`,
- keys that were regenerated after you copied them.

In [ ]:
# Ask Alpaca for your account details
account = trading_client.get_account()

# Alpaca sends numbers back as text (e.g. "100000"), so float(...) turns them into real numbers.
# In the print lines, {...:,.2f} means "add commas and show 2 decimal places", e.g. 100,000.00
print(f"Account number : {account.account_number}")
print(f"Status         : {account.status}")
print(f"Cash           : ${float(account.cash):,.2f}")
print(f"Buying power   : ${float(account.buying_power):,.2f}")  # how much you can spend on new trades
print(f"Equity         : ${float(account.equity):,.2f}")        # cash + value of everything you own

# Paper account numbers start with "PA". Warn if this one doesn't.
if not str(account.account_number).startswith("PA"):
    print("\n⚠️  This doesn't look like a paper account number. Double-check your keys!")

> **Formatting cheat sheet.** You'll see these inside `print(f"...")` throughout the notebook:
>
> | Code | Example output | Meaning |
> |---|---|---|
> | `{x:,.2f}` | `1,234.57` | commas, 2 decimal places |
> | `{x:.1%}` | `12.3%` | show a decimal (0.123) as a percent |
> | `{x:+.2%}` | `+12.35%` | same, with a + or − sign |
> | `{s:6}` | `AAPL  ` | pad text to 6 characters so columns line up |

---
## Part 6: The market clock

US stock markets are open **9:30 AM to 4:00 PM Eastern**, Monday to Friday, excluding holidays. Orders you submit outside those hours are **queued** until the next open instead of filling right away. That's the most common reason for "why didn't my order fill?"

In [ ]:
# Ask Alpaca whether the market is open right now
clock = trading_client.get_clock()

print(f"Market open right now? {clock.is_open}")
print(f"Current time           : {clock.timestamp}")
print(f"Next open              : {clock.next_open}")
print(f"Next close             : {clock.next_close}")

### Look up a stock
Before trading a symbol, you can check whether Alpaca supports it and whether you can buy **fractions** of a share.

In [ ]:
# Look up Apple's stock by its ticker symbol
asset = trading_client.get_asset("AAPL")

print(f"{asset.symbol}: {asset.name}")
print(f"  Exchange     : {asset.exchange}")      # where it's listed (e.g. NASDAQ)
print(f"  Tradable     : {asset.tradable}")      # can you trade it on Alpaca?
print(f"  Fractionable : {asset.fractionable}")  # can you buy part of a share?
print(f"  Shortable    : {asset.shortable}")     # can you bet against it (short sell)?

---
## Part 7: Market data

### A note on data feeds
The free Alpaca plan gives you:
- **IEX** feed: real-time data from the IEX exchange only (a small slice of all trading, but free and live).
- **SIP** feed: data from every exchange, but on the free plan you can't get the **most recent 15 minutes**.

We use `feed=DataFeed.IEX` everywhere to avoid permission errors.

### Historical bars
A **bar** (or candle) summarizes trading over one time period, like one day or one hour:
**open** (first price), **high**, **low**, **close** (last price), and **volume** (how many shares traded).

In [ ]:
from datetime import datetime, timedelta, timezone

from alpaca.data.enums import DataFeed
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

# Describe the data we want
request = StockBarsRequest(
    symbol_or_symbols=["AAPL", "SPY"],                         # which stocks
    timeframe=TimeFrame.Day,                                   # one bar per day
    start=datetime.now(timezone.utc) - timedelta(days=60),     # starting 60 days ago
    feed=DataFeed.IEX,                                         # use the free data feed
)

# Download it and turn it into a pandas table (a "DataFrame")
df = data_client.get_stock_bars(request).df

# Show the last 10 rows
df.tail(10)

The table holds both stocks together. Each row is labeled by **(symbol, date)**. To get just one stock, use `df.loc["SPY"]`:

In [ ]:
# Get just SPY's rows (.copy() gives us our own table we can safely add columns to)
spy = df.loc["SPY"].copy()

# Daily return = % change in closing price from one day to the next
spy["return"] = spy["close"].pct_change()

print(f"SPY bars: {len(spy)}")
print(f"Average daily return : {spy['return'].mean():.4%}")  # typical day's move
print(f"Daily volatility     : {spy['return'].std():.4%}")   # how much the daily moves vary (risk)

# Draw a line chart of the closing price
spy["close"].plot(title="SPY daily close (IEX)", figsize=(10, 4));

Want a different bar size? Swap `TimeFrame.Day` for `TimeFrame.Hour`, `TimeFrame.Minute`, or a custom size like 15 minutes:

```python
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
TimeFrame(15, TimeFrameUnit.Minute)
```

### Latest quote
A **quote** shows the current best **bid** (the most someone will pay) and **ask** (the least someone will sell for). The gap between them is the **spread**.

In [ ]:
from alpaca.data.requests import StockLatestQuoteRequest

# Ask for Apple's latest quote. The result is a dictionary, so ["AAPL"] picks out Apple's quote.
quote = data_client.get_stock_latest_quote(
    StockLatestQuoteRequest(symbol_or_symbols="AAPL", feed=DataFeed.IEX)
)["AAPL"]

print(f"AAPL bid: ${quote.bid_price:.2f}  ({quote.bid_size} shares wanted)")
print(f"AAPL ask: ${quote.ask_price:.2f}  ({quote.ask_size} shares offered)")
print(f"Spread  : ${quote.ask_price - quote.bid_price:.2f}")
print(f"As of   : {quote.timestamp}")

---
## Part 8: Placing orders

This is the main event. Every order goes through `trading_client.submit_order(...)`. You describe the order with an order *request* object, then submit it.

### Order types you'll use most

| Type | What it does | When to use |
|---|---|---|
| **Market** | Buy or sell **right now** at the best available price | You care about getting filled, not the exact price |
| **Limit** | Buy only at **your price or better** | You care about price, and are OK not getting filled |

### Time in force (how long an order stays active)
- `DAY`: cancelled automatically at market close if not filled.
- `GTC` ("good 'til cancelled"): stays open until it fills or you cancel it.

### 8.1 Market order: buy 1 share of SPY
If the market is closed, this order will show status `accepted` and fill at the next open.

In [ ]:
from alpaca.trading.enums import OrderSide, TimeInForce
from alpaca.trading.requests import MarketOrderRequest

# Describe the order...
order_request = MarketOrderRequest(
    symbol="SPY",                     # what to buy
    qty=1,                            # how many shares
    side=OrderSide.BUY,               # BUY or SELL
    time_in_force=TimeInForce.DAY,    # cancel at end of day if it hasn't filled
)

# ...then send it to Alpaca
market_order = trading_client.submit_order(order_request)

print(f"Order ID : {market_order.id}")      # a unique ID we can use to look the order up later
print(f"Status   : {market_order.status.value}")

`submit_order` returns as soon as Alpaca **receives** the order, which may be before it **fills** (actually buys the shares). To see what happened, look the order up again by its ID:

In [ ]:
import time

time.sleep(2)  # wait 2 seconds to give it time to fill (if the market is open)

# Get the latest info about our order
order = trading_client.get_order_by_id(market_order.id)

print(f"Status         : {order.status.value}")
print(f"Filled qty     : {order.filled_qty}")         # how many shares were bought
print(f"Filled avg px  : {order.filled_avg_price}")   # average price paid per share

### 8.2 Dollar-amount order: buy $500 of AAPL
Instead of a number of shares, you can say how many **dollars** to spend by using `notional` instead of `qty`. Alpaca buys fractional shares to match. This only works for stocks where `fractionable` is `True` (see Part 6) and only with `TimeInForce.DAY`.

In [ ]:
notional_order = trading_client.submit_order(MarketOrderRequest(
    symbol="AAPL",
    notional=500,                     # spend $500 (instead of qty=...)
    side=OrderSide.BUY,
    time_in_force=TimeInForce.DAY,    # dollar-amount orders must use DAY
))

print(f"Dollar-amount order status: {notional_order.status.value}")

### 8.3 Limit order
Here we place a buy limit order **10% below** the current price, so it almost certainly won't fill. That lets us practice checking and cancelling an open order.

In [ ]:
from alpaca.trading.requests import LimitOrderRequest

# Pick a price 10% below the current bid (quote comes from Part 7)
limit_price = round(quote.bid_price * 0.90, 2)   # round to whole cents

limit_order = trading_client.submit_order(LimitOrderRequest(
    symbol="AAPL",
    qty=1,
    side=OrderSide.BUY,
    time_in_force=TimeInForce.GTC,    # stay open until filled or cancelled
    limit_price=limit_price,          # only buy at this price or lower
))

print(f"Limit buy AAPL @ ${limit_price}: {limit_order.status.value}")

### 8.4 List your open orders

In [ ]:
from alpaca.trading.enums import QueryOrderStatus
from alpaca.trading.requests import GetOrdersRequest

# Ask for all orders that are still open (not yet filled or cancelled)
open_orders = trading_client.get_orders(filter=GetOrdersRequest(status=QueryOrderStatus.OPEN))

print(f"You have {len(open_orders)} open order(s):")
for o in open_orders:
    # .value turns labels like OrderSide.BUY into plain text like "buy"
    print(f"  {o.symbol}  {o.side.value}  {o.order_type.value}  status={o.status.value}")

### 8.5 Cancel an order

In [ ]:
# Cancel the limit order from 8.3 using its ID
trading_client.cancel_order_by_id(limit_order.id)

time.sleep(1)  # give Alpaca a second to process it
print("Limit order status:", trading_client.get_order_by_id(limit_order.id).status.value)

To cancel **every** open order at once, use `trading_client.cancel_orders()`.

---
## Part 9: Positions

A **position** is a stock you currently own. Once your SPY and AAPL orders fill (right away if the market is open, otherwise at the next open), they show up here.

In [ ]:
# Get everything you currently own
positions = trading_client.get_all_positions()

if len(positions) == 0:
    print("No open positions yet. If the market is closed, your orders fill at the next open.")

# Print one line per position
for p in positions:
    print(f"{p.symbol}:")
    print(f"   shares owned       : {float(p.qty):.4f}")
    print(f"   avg price paid     : ${float(p.avg_entry_price):,.2f}")
    print(f"   current value      : ${float(p.market_value):,.2f}")
    print(f"   unrealized P/L     : ${float(p.unrealized_pl):,.2f}")  # profit (+) or loss (-) if you sold now

### Close positions
`close_position("SPY")` sells all of your SPY. `close_all_positions(...)` sells everything.

Remove the `#` at the start of a line to "uncomment" it, then run the cell when you want to close out:

In [ ]:
# trading_client.close_position("SPY")                      # sell all of one stock
# trading_client.close_all_positions(cancel_orders=True)    # sell everything and cancel open orders

---
## Part 10: Things to know about paper trading

Paper trading is great for learning, but it's **more forgiving than reality**. Keep these in mind before trusting paper results:

- **No market impact.** Paper fills don't move the price. A real order for 10,000 shares would.
- **Optimistic fills.** Paper orders fill against the current quote; in real markets you may get partial fills or worse prices (slippage), especially in thin stocks.
- **No dividends or corporate-action cash** is simulated on paper.
- **Rate limits.** The API allows about **200 requests per minute** per account. Don't check on things in a tight loop; add `time.sleep(...)`.
- **Same rules as live.** Market hours, PDT, short-selling availability, and buying power all still apply.

### Useful links
- Alpaca dashboard: https://app.alpaca.markets
- `alpaca-py` docs: https://alpaca.markets/sdks/python/
- Trading API reference: https://docs.alpaca.markets/reference
- Paper trading docs: https://docs.alpaca.markets/docs/paper-trading

---
## Part 11: Try it yourself

Each exercise has two parts:

1. **A worked example.** Just run it and read the comments. Most examples package their code into a **function** (a reusable block of code started with `def`), so you can rerun it with different settings without rewriting anything.
2. **Now you try.** The cell is already started for you. Replace each **`FILL_IN`** with your own value, then run it.

> If you see `NameError: name 'FILL_IN' is not defined`, you missed one. Nothing here can break your account. It's all paper money, so experiment freely!

### 11.1 Portfolio snapshot
A function that prints your cash, your total account value, and what percentage of your account each stock makes up.

In [ ]:
def portfolio_summary():
    # 1. Get your account info (convert text to numbers with float)
    account = trading_client.get_account()
    equity = float(account.equity)   # total account value: cash + stocks
    cash = float(account.cash)

    print(f"Equity: ${equity:,.2f}")
    print(f"Cash:   ${cash:,.2f}")
    print()

    # 2. Get every stock you currently own
    positions = trading_client.get_all_positions()
    if len(positions) == 0:
        print("No open positions.")
        return   # nothing else to show, so stop here

    # 3. Print one line per stock
    for p in positions:
        value = float(p.market_value)   # what this position is worth right now
        percent = value / equity        # its share of your whole account
        print(f"{p.symbol:6} worth ${value:>10,.2f}   ({percent:.1%} of your account)")


# Run the function
portfolio_summary()

#### Now you try
Finish `portfolio_pnl()`, a version that shows **profit and loss** instead of portfolio weights. Fill in the two `FILL_IN`s.

**Hint:** each position `p` has `p.unrealized_pl` (profit/loss in dollars) and `p.unrealized_plpc` (profit/loss as a decimal, e.g. `0.05` = 5%).

In [ ]:
def portfolio_pnl():
    positions = trading_client.get_all_positions()
    total = 0   # we'll add up everyone's P/L here

    for p in positions:
        pl_dollars = float(FILL_IN)   # profit/loss in dollars
        pl_percent = float(FILL_IN)   # profit/loss as a decimal

        total = total + pl_dollars
        print(f"{p.symbol:6} P/L: ${pl_dollars:>9,.2f}  ({pl_percent:+.2%})")

    print(f"\nTotal unrealized P/L: ${total:,.2f}")


portfolio_pnl()

### 11.2 Compare stocks
A function that downloads prices for a few stocks and reports, for each one:
- **volatility**: how much the price bounces around from bar to bar (higher = riskier)
- **total return**: how much the price went up or down over the whole period

It also draws all of them on one chart, scaled so they each start at 1.0. That makes it easy to compare a $30 stock with a $500 one.

In [ ]:
import matplotlib.pyplot as plt


def compare_stocks(symbols, timeframe, days):
    # 1. Download price bars for all the symbols at once
    request = StockBarsRequest(
        symbol_or_symbols=symbols,
        timeframe=timeframe,
        start=datetime.now(timezone.utc) - timedelta(days=days),
        feed=DataFeed.IEX,
    )
    df = data_client.get_stock_bars(request).df

    # 2. Go through the stocks one at a time
    for symbol in symbols:
        closes = df.loc[symbol, "close"]                      # this stock's closing prices
        returns = closes.pct_change()                         # % change from each bar to the next
        volatility = returns.std()                            # how much those changes vary
        total_return = closes.iloc[-1] / closes.iloc[0] - 1   # last price vs. first price

        print(f"{symbol:6} volatility: {volatility:.3%}   total return: {total_return:+.2%}")

        # Add this stock to the chart, divided by its first price so it starts at 1.0
        (closes / closes.iloc[0]).plot(label=symbol, figsize=(10, 4))

    plt.title("Growth of each stock (starting value = 1.0)")
    plt.legend()
    plt.show()


# Compare three tech stocks using 1-hour bars over the last 30 days
compare_stocks(symbols=["AAPL", "MSFT", "NVDA"], timeframe=TimeFrame.Hour, days=30)

#### Now you try
Run your own comparison. Pick **three different stocks** and a **number of days**. This version uses **15-minute bars** instead of 1-hour bars.

Then look at the results: **was the most volatile stock also the one with the best return?**

In [ ]:
from alpaca.data.timeframe import TimeFrameUnit   # lets us make custom bar sizes

compare_stocks(
    symbols=[FILL_IN, FILL_IN, FILL_IN],              # three tickers in quotes, e.g. "TSLA", "KO", "AMZN"
    timeframe=TimeFrame(15, TimeFrameUnit.Minute),    # 15-minute bars
    days=FILL_IN,                                     # how far back to look, e.g. 10
)

### 11.3 Order babysitter
A function that:
1. places a limit buy a little **below** the current price,
2. checks on it every few seconds,
3. **cancels** it if it hasn't filled when time runs out.

Real trading bots manage orders this way: submit, check, then cancel or confirm.

> This cell takes up to a minute to run. If the market is closed, the order can't fill and will be cancelled at the end. That's expected.

In [ ]:
from alpaca.trading.enums import OrderStatus


def babysit_order(symbol, discount, check_every, max_wait):
    # 1. Get the current ask price (the cheapest price anyone is selling at)
    quote = data_client.get_stock_latest_quote(
        StockLatestQuoteRequest(symbol_or_symbols=symbol, feed=DataFeed.IEX)
    )
    ask = quote[symbol].ask_price

    # 2. Set our price a bit lower. discount=0.02 means 2% below the ask.
    limit_price = round(ask * (1 - discount), 2)

    # 3. Place the limit order for 1 share
    order = trading_client.submit_order(LimitOrderRequest(
        symbol=symbol,
        qty=1,
        side=OrderSide.BUY,
        time_in_force=TimeInForce.DAY,
        limit_price=limit_price,
    ))
    print(f"Placed limit buy for 1 {symbol} at ${limit_price} (ask was ${ask:.2f})")

    # 4. Check on the order every few seconds, until it fills or we run out of time
    seconds_waited = 0
    while seconds_waited < max_wait:
        time.sleep(check_every)                           # pause before checking again
        seconds_waited = seconds_waited + check_every
        order = trading_client.get_order_by_id(order.id)  # get the order's latest status
        print(f"  after {seconds_waited}s: {order.status.value}")

        if order.status == OrderStatus.FILLED:
            print(f"Filled at ${float(order.filled_avg_price):.2f} after {seconds_waited} seconds!")
            return   # we're done, so leave the function

    # 5. If we reach this line, the order never filled, so cancel it
    trading_client.cancel_order_by_id(order.id)
    print(f"Not filled after {max_wait} seconds, so it was cancelled.")


# Try to buy AAPL 2% below the ask, checking every 10 seconds, for up to 60 seconds
babysit_order(symbol="AAPL", discount=0.02, check_every=10, max_wait=60)

#### Now you try
A 2% discount usually won't fill in a minute. Run a version that's **more likely to fill**: pick a **different stock**, a **smaller discount**, and different timing.

*(The market has to be open for anything to fill. Check Part 6.)*

In [ ]:
babysit_order(
    symbol=FILL_IN,        # a ticker in quotes, e.g. "KO"
    discount=FILL_IN,      # e.g. 0.005 means 0.5% below the ask
    check_every=FILL_IN,   # seconds between checks, e.g. 5
    max_wait=FILL_IN,      # total seconds to wait, e.g. 90
)

### 11.4 A tiny strategy: moving-average crossover
A **moving average** is the average closing price over the last N days. It smooths out the day-to-day noise. A classic beginner strategy compares two of them:

- the **fast** average (e.g. last 10 days) reacts quickly to recent prices,
- the **slow** average (e.g. last 30 days) shows the longer trend.

**The rule:** if fast > slow, prices are trending up, so **own** the stock. Otherwise, **don't own** it.

The function checks the rule and makes at most one trade. With `dry_run=True` it only **prints** what it would do. Pass `dry_run=False` to actually place the order.

> This is for learning the mechanics, not investment advice. Later in the project we'll **backtest** strategies like this (test them on past data) before trusting them.

In [ ]:
import pandas as pd


def moving_average_strategy(symbol, fast, slow, dry_run=True):
    # 1. Download daily closing prices (slow * 2 calendar days is plenty of history)
    request = StockBarsRequest(
        symbol_or_symbols=symbol,
        timeframe=TimeFrame.Day,
        start=datetime.now(timezone.utc) - timedelta(days=slow * 2 + 10),
        feed=DataFeed.IEX,
    )
    closes = data_client.get_stock_bars(request).df.loc[symbol, "close"]

    # 2. Compute the two moving averages. .rolling(n).mean() = average of the last n values.
    fast_avg = closes.rolling(fast).mean()
    slow_avg = closes.rolling(slow).mean()

    # .iloc[-1] means "the most recent value"
    print(f"{symbol} last close: ${closes.iloc[-1]:.2f}")
    print(f"{fast}-day average: ${fast_avg.iloc[-1]:.2f}    {slow}-day average: ${slow_avg.iloc[-1]:.2f}")

    # 3. What does the rule say, and what do we own right now?
    should_own = fast_avg.iloc[-1] > slow_avg.iloc[-1]
    owned_symbols = [p.symbol for p in trading_client.get_all_positions()]
    own_it = symbol in owned_symbols
    print(f"Rule says own it? {should_own}    Own it now? {own_it}")

    # 4. Trade only if what we own doesn't match what the rule says
    if should_own and not own_it:
        print(f"-> BUY 1 share of {symbol}")
        if not dry_run:
            trading_client.submit_order(MarketOrderRequest(
                symbol=symbol, qty=1, side=OrderSide.BUY, time_in_force=TimeInForce.DAY,
            ))
    elif own_it and not should_own:
        print(f"-> SELL all {symbol}")
        if not dry_run:
            trading_client.close_position(symbol)
    else:
        print("-> No trade needed")

    if dry_run:
        print("(dry run: no order was placed)")

    # 5. Chart the price and both averages
    pd.DataFrame({
        "close": closes,
        f"{fast}-day avg": fast_avg,
        f"{slow}-day avg": slow_avg,
    }).plot(title=f"{symbol} moving averages", figsize=(10, 4))
    plt.show()


# Check the rule for SPY with a 10-day and 30-day average (dry run, so no trade)
moving_average_strategy(symbol="SPY", fast=10, slow=30)

#### Now you try
Make your own version: pick a **different symbol** (e.g. `"QQQ"` or a single stock) and **different averages** (e.g. 5 and 20 days). Keep it as a dry run first.

**Does your version give a different answer than the SPY example?** When you're happy with it, add `dry_run=False` to place the trade for real (paper money, of course).

In [ ]:
moving_average_strategy(
    symbol=FILL_IN,   # a ticker in quotes, e.g. "QQQ"
    fast=FILL_IN,     # the shorter average in days, e.g. 5
    slow=FILL_IN,     # the longer average in days, e.g. 20 (must be bigger than fast)
)

### 11.5 Clean up
Cancel every open order and sell everything, so your account starts fresh.

> If the market is closed, the sell orders are queued and your positions will still show up until the next open.

In [ ]:
# Cancel every open order
cancelled = trading_client.cancel_orders()
print(f"Cancelled {len(cancelled)} open order(s).")

# Sell every position (cancel_orders=True also cancels anything still pending)
closed = trading_client.close_all_positions(cancel_orders=True)
print(f"Placed sell orders for {len(closed)} position(s).")

#### Now you try
Instead of selling everything, sell **only the stocks that are losing money** and keep the winners. Fill in the two `FILL_IN`s.

*(If the example above already sold everything, first buy a few small positions: remove the `#` from the setup lines at the top of the cell, run it, and wait for the orders to fill while the market is open.)*

In [ ]:
# --- Optional setup: buy $100 each of a few stocks so you have something to work with ---
# for s in ["AAPL", "KO", "XOM"]:
#     trading_client.submit_order(MarketOrderRequest(
#         symbol=s, notional=100, side=OrderSide.BUY, time_in_force=TimeInForce.DAY))

for p in trading_client.get_all_positions():
    pl = float(p.unrealized_pl)   # profit/loss in dollars (negative means losing money)

    if FILL_IN:                                # a condition that's True when losing, e.g.  pl < 0
        trading_client.close_position(FILL_IN) # which stock to sell? (hint: p.symbol)
        print(f"Sold {p.symbol} (P/L ${pl:,.2f})")
    else:
        print(f"Kept {p.symbol} (P/L ${pl:,.2f})")

---
## 🎉 You're done!

You now have a working Alpaca paper account and know how to pull data, place and manage orders, and run a simple strategy. Keep your notebook around; we'll build on these pieces throughout the project.